# 参数管理

对于单个隐藏层的MLP，测试一下

In [1]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[0.2807],
        [0.2476]], grad_fn=<AddmmBackward0>)

## 参数访问

对于Sequential类定义的模型，我们可以通过索引来访问模型的任意层，如下所示，检查第二个全连接层的参数；

net是一个有序的字典(OrderedDict)

In [ ]:
print(net[2].state_dict())  

OrderedDict([('weight', tensor([[ 0.0640, -0.1601,  0.2181,  0.3186,  0.0094, -0.1631,  0.2929,  0.2987]])), ('bias', tensor([0.0801]))])


### 目标参数

每个参数都表示为参数类的一个实例，要对参数执行任何操作，首先需要访问底层的数值

In [ ]:
print(type(net[2].bias))
print(net[2].bias)  # 这里还包括梯度信息（计算图）
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.0801], requires_grad=True)
tensor([0.0801])


### 一次性访问所有参数

对所有参数进行操作时，递归整个树来提取每个字块的参数

In [4]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


两个注意点：
* ```named_parameters()``` 方法只会返回网络中需要训练的参数（如权重和偏置）。而ReLU是一个无参数的激活函数，它只是对输入进行简单的数学变换,ReLU层不包括任何需要学习的权重或偏置。
* 在PyTorch中，```nn.Linear(in_features, out_features)```的权重矩阵形状是 ```(out_features, in_features)```。
因为Pytorch中线性层的计算过程如下所示：
$$
y = x W^T + b
$$

这里公式*和书上不一样*

也提供了另外一种访问参数的方式，如下所示：
```
net.state_dict()['2.bias].data
```

### 从嵌套块收集参数

把多个块互相嵌套，观察其工作流程

In [ ]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())  # 这里是为了有序字典的key是字符串形式
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))  # 三层sequential
rgnet(X)

tensor([[-0.0961],
        [-0.0961]], grad_fn=<AddmmBackward0>)

In [6]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


分层嵌套也可以通过嵌套列表索引去访问它们，下面访问第一个主要的块中第二个子块的第一层的偏置项

In [7]:
rgnet[0][1][0].bias.data

tensor([ 0.4180, -0.3439,  0.2583, -0.4743,  0.3359, -0.2810,  0.2278, -0.2984])

## 参数初始化

### 内置初始化

调用内置的初始化器，把所有的权重参数都初始化为标准差为0.01的高斯随机变量，偏置参数设置为0

In [ ]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)  # 原地操作
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0026,  0.0020, -0.0150,  0.0045]), tensor(0.))

也可以将参数初始化为给定的常数，下面初始化为1

In [9]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

也可以调用Xavier初始化方法，下代码将第一个神经网络层初始化为Xavier分布，第三个神经网络层初始化为常量42

In [10]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.5732,  0.5073, -0.0612, -0.0853])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


### 自定义初始化

见书p202-p203

## 参数绑定

我们希望可以在多个层之间共享参数：定义一个稠密层，使用他的参数来设置另一个层的参数

In [11]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),  # 两个shared指向同一块内存
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


如果是反向传播的话，*两个shared的grad会累加*